# Rents, vacancy and rental-market tightness: Moncton and Saint John (2020 to today)

**Question**: how have rents and vacancy rates changed in Moncton and Saint John since 2020, and what does this reveal about the tightness of the rental market in the two largest cities of southern New Brunswick?

Plan: 1) load the data, 2) rents, 3) vacancy rates, 4) rent growth, 5) tightness summary, 6) findings and limitations.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the src/ folder importable from the notebook
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.analysis import (
    BALANCED_VACANCY_PCT,
    add_rent_growth,
    add_tightness_indicators,
    change_since,
    load_tidy,
    tightness_summary,
)

BASE_YEAR = 2020
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.figsize"] = (9, 5)

## 1. Data

The file `data/processed/housing_tidy.csv` (one row per city and year) is produced by `python -m src.build_tidy` from the two raw Statistics Canada files placed in `data/raw/`:

- table 34-10-0133-01: average rents (structure type kept: row and apartment structures of three units and over; 2 bedrooms);
- table 34-10-0130-01: vacancy rates (census metropolitan areas). The row for **all CMAs together** is used as a benchmark.

Columns: `city, year, avg_rent_2br, vacancy_rate, all_cma_vacancy_rate`.

In [ ]:
df = load_tidy()
print(df.shape)
df.head(12)

In [ ]:
# Quality checks: missing values, duplicates, years covered
print(df.isna().sum())
print("Duplicate city/year rows:", df.duplicated(["city", "year"]).sum())
print(df.groupby("city")["year"].agg(["min", "max", "count"]))

## 2. Rent evolution

In [ ]:
rents = df.pivot(index="year", columns="city", values="avg_rent_2br")
ax = rents.plot(marker="o")
ax.set_title("Average monthly rent, 2-bedroom apartment")
ax.set_ylabel("$ per month")
ax.set_xlabel("Year")
ax.legend(title="City")
plt.savefig(FIGURES / "rent_2_bedroom.png", dpi=150, bbox_inches="tight")
plt.show()
rents

In [ ]:
latest = df["year"].max()
chg = change_since(df, "avg_rent_2br", BASE_YEAR)
summary = chg[chg["year"] == latest][["city", "avg_rent_2br", "avg_rent_2br_change_pct"]].copy()
base = df[df["year"] == BASE_YEAR].set_index("city")["avg_rent_2br"]
summary["increase_in_dollars"] = summary["avg_rent_2br"] - summary["city"].map(base)
summary.round(1)

In [ ]:
# Rents by number of bedrooms (2020 vs latest year)
b = pd.read_csv(ROOT / "data" / "processed" / "rents_by_bedroom.csv")
bp = b[b["year"].isin([BASE_YEAR, latest])].pivot_table(
    index=["city", "bedrooms"], columns="year", values="avg_rent"
)
bp["change_pct"] = (bp[latest] / bp[BASE_YEAR] - 1) * 100
bp.round(1)

## 3. Vacancy rates

Two references are used to judge whether a vacancy rate is low:
- the vacancy rate of **all CMAs together**, from the same table (no convention needed);
- a **3% rule of thumb** for a balanced rental market (a convention, not an official threshold).

In [ ]:
vac = df.pivot(index="year", columns="city", values="vacancy_rate")
bench = df.groupby("year")["all_cma_vacancy_rate"].first()

ax = vac.plot(marker="o")
bench.plot(ax=ax, color="grey", linestyle="--", label="All CMAs")
ax.axhline(BALANCED_VACANCY_PCT, color="red", linestyle=":", label="3% rule of thumb")
ax.set_title("Rental vacancy rate")
ax.set_ylabel("%")
ax.set_xlabel("Year")
ax.legend(title="")
plt.savefig(FIGURES / "vacancy_rate.png", dpi=150, bbox_inches="tight")
plt.show()

tight = add_tightness_indicators(df)
tight.pivot(index="year", columns="city", values="vacancy_gap_pts").round(1)

## 4. Rent growth, year over year

In [ ]:
growth = add_rent_growth(df)
g = growth.pivot(index="year", columns="city", values="rent_growth_pct").dropna()
ax = g.plot.bar()
ax.set_title("Year-over-year growth of the average 2-bedroom rent")
ax.set_ylabel("%")
ax.set_xlabel("Year")
ax.legend(title="City")
plt.xticks(rotation=0)
plt.savefig(FIGURES / "rent_growth.png", dpi=150, bbox_inches="tight")
plt.show()
g.round(1)

## 5. Tightness summary

In [ ]:
tightness_summary(df, BASE_YEAR).round(1).T

## 6. Findings and limitations

**Findings (data up to 2025)**
- Rents rose a lot in both cities: **+53.0% in Moncton** ($949 to $1,452) and **+56.4% in Saint John** ($825 to $1,290), about 9% a year on average.
- Both markets were **tight for most of the period**: the vacancy rate was below 3% in 5 of the 6 years in Moncton and 4 of the 6 in Saint John, and Moncton's low was 1.2% in 2023.
- In 2025 **rent growth slowed in both cities** (Moncton +6.4%, Saint John +5.0%, each the lowest of the period), but **vacancy signals diverge**. Moncton's vacancy rate jumped to 3.8%, above the all-CMA benchmark (3.1%), a first sign of easing. Saint John's vacancy rate is back to 2.1%, below the benchmark, after a one-year rise to 3.9% in 2024.
- Rents kept rising even in the year a city's vacancy rate went above 3% (Saint John in 2024: vacancy 3.9%, rent +12.4%), so a higher vacancy rate does not by itself mean rents are falling.

**Limitations**
- CMHC average rent covers existing units, not the asking rent of new listings.
- Only six yearly observations per city: the evidence describes a pattern but cannot support statistical tests or causes.
- Vacancy and rents come from an October survey; they do not show changes within the year.
- A single unit type (2-bedroom) is used for the main comparison; other sizes are in `rents_by_bedroom.csv`.
- The 3% threshold is a rule of thumb: cite a source before relying on it.
- Affordability (rent relative to income) is not covered: it needs income data by city.